In [1]:
import spatialdata as sd
import napari_spatialdata as nsd
import squidpy as sq
import numpy as np
import pandas as pd
import zarr
from spatialdata.transformations.transformations import Affine, Identity, Scale
from spatialdata.models import Image2DModel
from dask_image.imread import imread

In [2]:
sdata = sd.read_zarr("Version 2 Zarr//Sample_4D_annotated.zarr")

In [3]:
df = sdata.points["transcripts"].compute()

In [4]:
df.rename(columns={"x": "x_location", "y": "y_location", "z": "z_location"}, inplace=True)

In [5]:
df

,x_location,y_location,z_location,feature_name,cell_id,nucleus_distance,fov_name,overlaps_nucleus,transcript_id,qv
0,185.650269,2248.806641,24.312099,VPS33A,UNASSIGNED,592.194336,Y3,0,281861523767305,38.604805
1,142.769318,2221.099121,24.420168,SNHG14,UNASSIGNED,643.245972,Y3,0,281861523767335,20.623095
2,70.767570,2198.220459,24.776201,ZEB2,UNASSIGNED,716.256531,Y3,0,281861523767359,8.692277
3,53.393982,2239.417725,24.498346,ITGB1,UNASSIGNED,710.744873,Y3,0,281861523767366,16.679827
4,207.748810,2174.929199,24.443127,SPECC1L,UNASSIGNED,618.756897,Y3,0,281861523767369,40.000000
...,...,...,...,...,...,...,...,...,...,...
1654569,9043.876953,4341.580566,11.861894,ANTXR1,UNASSIGNED,595.814697,AA17,0,281500746539200,40.000000
1654570,9045.340820,4314.265625,11.602382,MAF,UNASSIGNED,583.554382,AA17,0,281500746540052,40.000000
1654571,9018.395508,4304.219727,11.942560,ITGA6,UNASSIGNED,555.095703,AA17,0,281500746540777,40.000000
1654572,9037.218750,4270.228027,11.845038,S100A11,UNASSIGNED,556.493225,AA17,0,281500746540779,25.047544


In [6]:
df["cell_id"].unique()

array(['UNASSIGNED', 'jhkgmakd-1', 'jhkbpiil-1', ..., 'eeeplkkm-1',
       'gcdakifn-1', 'agbmaffa-1'], dtype=object)

In [7]:
# Define the conversion function
def convert_to_int(val):
    if val == 'UNASSIGNED':
        return 0
    return unique_mapping[val]

# Get unique values in the column (excluding 'UNASSIGNED')
unique_values = df['cell_id'].unique()
unique_values = unique_values[unique_values != 'UNASSIGNED']

# Create a mapping from unique values to integers starting from 1
unique_mapping = {val: i+1 for i, val in enumerate(unique_values)}

# Apply the conversion
df['cell_id'] = df['cell_id'].apply(convert_to_int)

In [8]:
df

,x_location,y_location,z_location,feature_name,cell_id,nucleus_distance,fov_name,overlaps_nucleus,transcript_id,qv
0,185.650269,2248.806641,24.312099,VPS33A,0,592.194336,Y3,0,281861523767305,38.604805
1,142.769318,2221.099121,24.420168,SNHG14,0,643.245972,Y3,0,281861523767335,20.623095
2,70.767570,2198.220459,24.776201,ZEB2,0,716.256531,Y3,0,281861523767359,8.692277
3,53.393982,2239.417725,24.498346,ITGB1,0,710.744873,Y3,0,281861523767366,16.679827
4,207.748810,2174.929199,24.443127,SPECC1L,0,618.756897,Y3,0,281861523767369,40.000000
...,...,...,...,...,...,...,...,...,...,...
1654569,9043.876953,4341.580566,11.861894,ANTXR1,0,595.814697,AA17,0,281500746539200,40.000000
1654570,9045.340820,4314.265625,11.602382,MAF,0,583.554382,AA17,0,281500746540052,40.000000
1654571,9018.395508,4304.219727,11.942560,ITGA6,0,555.095703,AA17,0,281500746540777,40.000000
1654572,9037.218750,4270.228027,11.845038,S100A11,0,556.493225,AA17,0,281500746540779,25.047544


In [9]:
df["cell_id"].unique()

array([    0,     1,     2, ..., 24659, 24660, 24661], dtype=int64)

In [10]:
df.to_csv("segmentation_masks//4D_transcripts.csv", index=False)